<a href="https://colab.research.google.com/github/kavya-83175/24B11DS151_DriveReady_OPC_2026/blob/main/FinMate_AI_Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install groq

In [2]:
import os
import json
import functools

from groq import Groq
from groq import RateLimitError, APIConnectionError
from google.colab import userdata

In [4]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq()

In [5]:
messages = [
    {"role": "system", "content": "You are FinMate AI."},
    {"role": "user", "content": "What is my daily income?"},
    {"role": "assistant", "content": "Please provide your sales records."},
    {"role": "user", "content": "I earned Rs 8000 today."}
]

print(messages)

[{'role': 'system', 'content': 'You are FinMate AI.'}, {'role': 'user', 'content': 'What is my daily income?'}, {'role': 'assistant', 'content': 'Please provide your sales records.'}, {'role': 'user', 'content': 'I earned Rs 8000 today.'}]


In [6]:
SYSTEM = """
ROLE:
You are FinMate AI, a friendly finance assistant
for small retail shop owners.

FACTS:
You help shop owners understand income, expenses,
sales and basic financial concepts.

RULES:
Never invent financial information.
Never assume missing transaction records.
Ask for clarification when information is incomplete.
Never claim a profit figure without actual data.

FORMAT:
Reply politely and clearly in under 60 words.
"""

In [7]:
messages = [
    {
        "role": "system",
        "content": """
        Classify the user's message into one word:
        income, expense or question.
        """
    },

    {"role": "user", "content": "I earned Rs 5000 today."},
    {"role": "assistant", "content": "income"},

    {"role": "user", "content": "I paid Rs 1000 for electricity."},
    {"role": "assistant", "content": "expense"},

    {"role": "user", "content": "How can I calculate my profit?"},
    {"role": "assistant", "content": "question"},

    {"role": "user", "content": "I purchased stock for Rs 3000."}
]

In [8]:
{
    "type": "income",
    "amount": 8000,
    "category": "sales",
    "description": "Daily sales"
}

{'type': 'income',
 'amount': 8000,
 'category': 'sales',
 'description': 'Daily sales'}

In [9]:
reply = '{"type": "income", "amount": 8000, "category": "sales"}'

transaction = json.loads(reply)

print(transaction["type"])
print(transaction["amount"])

income
8000


In [10]:
history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8

In [11]:
def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=recent,
        temperature=0.3
    )

    reply = response.choices[0].message.content

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [13]:
print(chat("My shop earned Rs 8000 today."))

print(chat("I spent Rs 2000 purchasing stock."))

print(chat("What did I tell you about my sales?"))

print(chat("My shop is related to clothes how can I improve my profits?"))

You told me that your shop’s sales (gross revenue) for today were **Rs 8,000**.
Got it!  

- Sales (gross revenue): Rs 8,000  
- Stock purchase expense: Rs 2,000  

Net profit so far: **Rs 6,000**.  

If you have any other costs today (rent, wages, utilities, etc.), let me know to finalize the profit.
You said your shop’s sales (gross revenue) for today were **Rs 8,000**.
Here are quick ways a clothing shop can lift profit:

1. **Track best‑selling styles** – order more of what sells fast, cut slow‑moving items.  
2. **Bundle looks** (e.g., shirt + belt) to raise average ticket.  
3. **Run limited‑time promos** on excess stock to free space.  
4. **Negotiate bulk discounts** with suppliers.  
5. **Use eye‑catching displays** and clear pricing to boost impulse buys.  
6. **Collect customer emails** for loyalty offers and repeat visits.  

Let me know which area you’d like to explore first!


In [14]:
# for practice
MENU = {
    "rice": 60,
    "sugar": 45,
    "oil": 150,
    "soap": 40
}

In [15]:
print(MENU["rice"])

print(MENU.get("milk", "Product not found"))

60
Product not found


In [16]:
text = json.dumps(MENU)

print(text)

back = json.loads(text)

print(back)

{"rice": 60, "sugar": 45, "oil": 150, "soap": 40}
{'rice': 60, 'sugar': 45, 'oil': 150, 'soap': 40}


In [17]:
def get_price(item: str, qty: int = 1) -> int:
    """
    Return the total price in rupees
    for the requested product quantity.
    """
    return MENU[item.lower().strip()] * qty

In [18]:
print(get_price("rice", 5))
print(get_price("soap", 3))

300
120


In [19]:
def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        print("Calling:", func.__name__)

        result = func(*args, **kwargs)

        print("Result:", result)

        return result

    return wrapper

In [20]:
@log_call
def get_price(item: str, qty: int = 1) -> int:
    """
    Calculate the total product price.
    """
    return MENU[item.lower().strip()] * qty

In [21]:
get_price("rice", 5)

Calling: get_price
Result: 300


300

In [22]:
get_price = log_call(get_price)

In [23]:
try:
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=history
    )

    print(response.choices[0].message.content)

except RateLimitError:
    print("FinMate is busy. Please try again shortly.")

except APIConnectionError:
    print("Unable to connect. Please try again.")

Here are quick profit‑boosters for a clothing shop:

1. Track top‑selling styles and stock more of them.  
2. Bundle items (e.g., shirt + belt) to raise the average ticket.  
3. Offer flash promos on slow‑moving stock.  
4. Negotiate bulk discounts with suppliers.  
5. Use attractive displays and clear pricing.  

Which would you like to try first?


In [24]:
print("Welcome to FinMate AI!")

while True:
    text = input("You: ")

    if text.lower().strip() in ("bye", "exit", "quit"):
        print("FinMate: Thank you for using FinMate AI!")
        break

    print("FinMate:", chat(text))

Welcome to FinMate AI!
You: Hi
FinMate: Hello! 👋 How can I help you today with your shop’s finances or profit‑boosting ideas?
You: I am having cloth shop can you generate my monthly sales?
FinMate: I’d love to help, but I need the actual numbers first. Could you share each day’s sales (or a weekly total) for the month, plus any major expenses? With that data I can calculate your monthly revenue and profit.
You: ok bye
FinMate: Take care! If you need any financial help later, just let me know. Happy selling!
You: bye
FinMate: Thank you for using FinMate AI!
